# 🏆 XIANGQI-RIM: HỆ THỐNG KHAI THÁC CỜ SẠCH & HUẤN LUYỆN AI PHÂN TÁN CỘNG ĐỒNG
Welcome to **Xiangqi-RIM Distributed Community Data Miner & Trainer**!

### 🌟 Tầm Nhìn Dự Án
Góp sức GPU/CPU của bạn để tự động sinh hàng trăm ngàn ván cờ **100% SẠCH & CHUẨN LUẬT CỜ TƯỚNG** bằng Native Rust Engine. Toàn bộ dữ liệu sinh ra sẽ được hợp nhất tự động và đẩy lên **HuggingFace Hub** (`hoduyquocbao/xiangqi-r1-dataset`) để huấn luyện các thế hệ **NNUE AI đỉnh cao** (+300 đến +500 ELO)!

> ⚠️ **CẤU HÌNH TOKEN**: Vào 🔑 Secrets (thanh bên trái Colab) → Thêm key `HF_TOKEN` với giá trị write token từ https://huggingface.co/settings/tokens

In [ ]:
# 1. Clone Git Repository & Install Environment
!git clone https://github.com/hoduyquocbao/xiangqi-rim.git
%cd xiangqi-rim

!pip install -q gradio huggingface_hub torch numpy
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] += ":" + os.path.expanduser("~/.cargo/bin")

# Đọc token từ Colab Secrets (KHÔNG hardcode trong mã nguồn)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    print('⚠️ HF_TOKEN chưa được cấu hình — dữ liệu chỉ lưu cục bộ, không upload.')
    print('   → Vào 🔑 Secrets → Thêm key HF_TOKEN → Tạo tại https://huggingface.co/settings/tokens')
else:
    print(f'✅ HF_TOKEN sẵn sàng ({HF_TOKEN[:8]}...)')

print("✅ Rust environment ready!")
!cargo --version

In [ ]:
# 2. Build Native Rust Data Miner (Release Mode)
!cargo build --release --example 20_parallel_mine

In [ ]:
# 3. Launch Gradio Distributed Miner GUI (Non-blocking Web Interface)
from scripts.community_miner_gradio import create_ui

print("=" * 60)
print("🌐 BƯỚC 3: KHỞI CHẠY INTERACTIVE GRADIO MINER GUI")
print("=" * 60)
print("💡 Giao diện Web GUI sẽ chạy ở chế độ non-blocking.")
print("   Bạn có thể sử dụng Web GUI để khai thác dữ liệu cờ Tướng trực tiếp trên trình duyệt,")
print("   HOẶC tiếp tục chạy Cell 4 bên dưới để huấn luyện mô hình NNUE trên GPU T4.")

demo = create_ui()
demo.queue().launch(share=True, prevent_thread_lock=True)
print("🚀 Gradio Community Data Miner Launched Successfully!")


In [ ]:
# 4. GPU T4 Heavy NNUE Model Training (Train on All Community Data)
import os
import json
import time
import struct
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from huggingface_hub import HfApi, hf_hub_download

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("============================================================")
print("🚀 GPU T4 HEAVY TRAINING ON CROWDSOURCED COMMUNITY DATASET")
print("============================================================")
print("  • Device:", device)
if device.type == "cuda":
    print("  • GPU Model:", torch.cuda.get_device_name(0))

repo_id = "hoduyquocbao/xiangqi-r1-dataset"

# Download latest clean dataset (public — không cần token để đọc)
# Ưu tiên kiểm tra tệp dữ liệu vừa mine cục bộ trong data/ hoặc /content/
local_mined = [f for f in glob.glob("data/*.jsonl") + glob.glob("/content/*.jsonl") if os.path.exists(f) and os.path.getsize(f) > 0]
if local_mined:
    clean_dataset_path = local_mined[0]
    print(f"📁 Tìm thấy dữ liệu mine cục bộ: {clean_dataset_path} ({os.path.getsize(clean_dataset_path)/1024/1024:.1f} MB)")
else:
    print(f"📥 Tải dataset mới nhất từ HuggingFace Hub ({repo_id})...")
    clean_dataset_path = hf_hub_download(
        repo_id=repo_id,
        filename="data/selfplay_samples_gen5.jsonl",
        repo_type="dataset",
        token=HF_TOKEN if HF_TOKEN else None
    )

PIECE_MAP = {
    'R': 0, 'N': 1, 'B': 2, 'A': 3, 'K': 4, 'C': 5, 'P': 6,
    'r': 7, 'n': 8, 'b': 9, 'a': 10, 'k': 11, 'c': 12, 'p': 13
}

def flip_sq(sq):
    return (9 - (sq // 9)) * 9 + (sq % 9)

def get_feature_index(king, piece, square, side, view):
    if side != view:
        piece = piece + 7 if piece < 7 else piece - 7
        king = flip_sq(king)
        square = flip_sq(square)

    file = king % 9
    rank = king // 9
    tf = square % 9
    tr = square // 9

    if file > 4:
        norm = rank * 9 + (8 - file)
        target = tr * 9 + (8 - tf)
    else:
        norm = king
        target = square

    col = norm % 9
    row = norm // 9
    base = row * 5 + col
    return base * 1260 + piece * 90 + target

def validate_fen(fen):
    """Kiểm tra tính hợp lệ cơ bản của FEN cờ tướng."""
    parts = fen.split()
    if len(parts) < 2:
        return False
    rows = parts[0].split('/')
    if len(rows) != 10:
        return False
    board = parts[0]
    if 'K' not in board or 'k' not in board:
        return False
    for row in rows:
        count = 0
        for ch in row:
            if ch.isdigit():
                count += int(ch)
            else:
                count += 1
        if count != 9:
            return False
    return True

def parse_fen(fen):
    grid = [None] * 90
    parts = fen.split()
    rows = parts[0].split('/')
    king_red = 85
    king_black = 4
    r = 0
    for row in rows:
        c = 0
        for ch in row:
            if ch.isdigit(): c += int(ch)
            else:
                sq = r * 9 + c
                grid[sq] = ch
                if ch == 'K': king_red = sq
                elif ch == 'k': king_black = sq
                c += 1
        r += 1
    return grid, king_red, king_black

# Nạp dữ liệu với Validation Gateway
samples = []
rejected = 0
with open(clean_dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip(): continue
        try:
            data = json.loads(line)
            fen = data.get("fen", "")
            # Hỗ trợ cả 'score' và 'eval' (thống nhất field name)
            score = float(data.get("score", data.get("eval", 0)))
            # Validation Gateway: kiểm tra FEN hợp lệ và score trong phạm vi
            if not validate_fen(fen):
                rejected += 1
                continue
            if abs(score) > 30000:
                rejected += 1
                continue
            grid, kr, kb = parse_fen(fen)
            rf, bf = [], []
            for sq, p in enumerate(grid):
                if p is None: continue
                p_code = PIECE_MAP[p]
                rf.append(get_feature_index(kr, p_code, sq, 0, 0))
                bf.append(get_feature_index(kb, p_code, sq, 1, 1))
            samples.append((rf, bf, score / 400.0))
        except Exception: 
            rejected += 1

print(f"✅ Parsed {len(samples):,} Clean Samples! (Rejected: {rejected:,})")

# Train/Test Split 80/20 để phát hiện overfitting
random.shuffle(samples)
split = int(len(samples) * 0.8)
train_samples = samples[:split]
test_samples = samples[split:]
print(f"📊 Train: {len(train_samples):,} | Test: {len(test_samples):,}")

class NNUE_Community(nn.Module):
    def __init__(self):
        super().__init__()
        self.ft = nn.EmbeddingBag(65536, 256, mode="sum", sparse=False)
        self.hidden = nn.Linear(512, 32)
        self.output = nn.Linear(32, 1)
        nn.init.kaiming_normal_(self.hidden.weight)
        nn.init.kaiming_normal_(self.output.weight)

    def forward(self, r_idx, r_off, b_idx, b_off):
        r_acc = torch.clamp(self.ft(r_idx, r_off), 0.0, 1.0)
        b_acc = torch.clamp(self.ft(b_idx, b_off), 0.0, 1.0)
        both = torch.cat([r_acc, b_acc], dim=1)
        h = torch.clamp(self.hidden(both), 0.0, 1.0)
        return self.output(h)

def prepare_batch(batch, device):
    r_flat, r_off = [], [0]
    b_flat, b_off = [], [0]
    y_list = []
    for rf, bf, score in batch:
        r_flat.extend(rf)
        r_off.append(len(r_flat))
        b_flat.extend(bf)
        b_off.append(len(b_flat))
        y_list.append(score)
    r_t = torch.tensor(r_flat, dtype=torch.long, device=device)
    ro_t = torch.tensor(r_off[:-1], dtype=torch.long, device=device)
    b_t = torch.tensor(b_flat, dtype=torch.long, device=device)
    bo_t = torch.tensor(b_off[:-1], dtype=torch.long, device=device)
    y_t = torch.tensor(y_list, dtype=torch.float32, device=device).unsqueeze(1)
    return r_t, ro_t, b_t, bo_t, y_t

def evaluate_set(model, dataset, device, batch_size=2048):
    """Đánh giá MSE trên tập dữ liệu (train hoặc test)."""
    model.eval()
    total_loss = 0.0
    nb = 0
    crit = nn.MSELoss()
    with torch.no_grad():
        for i in range(0, len(dataset), batch_size):
            b = dataset[i:i+batch_size]
            r_t, ro_t, b_t, bo_t, y_t = prepare_batch(b, device)
            pred = model(r_t, ro_t, b_t, bo_t)
            loss = crit(pred, y_t)
            total_loss += loss.item()
            nb += 1
    return total_loss / max(1, nb)

model = NNUE_Community().to(device)
opt = optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=300)
crit = nn.MSELoss()

EPOCHS = 300
BATCH_SIZE = 2048
best_test_loss = float('inf')
patience = 0
MAX_PATIENCE = 30  # Early stopping sau 30 epochs không cải thiện

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

print(f"🔥 Training NNUE Model on GPU T4 ({EPOCHS} Epochs, Early Stopping patience={MAX_PATIENCE})...")
pbar = tqdm(range(1, EPOCHS + 1), desc="Training NNUE Epochs", unit="epoch")
for epoch in pbar:
    model.train()
    random.shuffle(train_samples)
    total_loss = 0.0
    nb = 0
    for i in range(0, len(train_samples), BATCH_SIZE):
        b = train_samples[i:i+BATCH_SIZE]
        r_t, ro_t, b_t, bo_t, y_t = prepare_batch(b, device)
        opt.zero_grad()
        pred = model(r_t, ro_t, b_t, bo_t)
        loss = crit(pred, y_t)
        loss.backward()
        opt.step()
        total_loss += loss.item()
        nb += 1
    sched.step()
    
    # Đánh giá trên tập test mỗi 10 epoch
    if epoch % 10 == 0 or epoch == 1 or epoch == EPOCHS:
        train_mse = total_loss / max(1, nb)
        test_mse = evaluate_set(model, test_samples, device)
        train_mae = (train_mse ** 0.5) * 400.0
        test_mae = (test_mse ** 0.5) * 400.0
        improved = '✅' if test_mse < best_test_loss else '⚠️'
        pbar.set_postfix({"Train_MSE": f"{train_mse:.5f}", "Test_MSE": f"{test_mse:.5f}", "Test_MAE": f"{test_mae:.1f}cp"})
            print(f"  [GPU] Epoch {epoch:3d}/{EPOCHS:3d} | Train MSE: {train_mse:.6f} ({train_mae:.1f}cp) | Test MSE: {test_mse:.6f} ({test_mae:.1f}cp) {improved}")
        
        if test_mse < best_test_loss:
            best_test_loss = test_mse
            patience = 0
        else:
            patience += 10
        
        if patience >= MAX_PATIENCE:
            print(f"  ⏹️ Early Stopping tại Epoch {epoch} (Test loss không cải thiện trong {MAX_PATIENCE} epochs)")
            break

# Xuất trọng số nhị phân format XRNN
out_path = "nnue_weights_community.bin"
with open(out_path, "wb") as f:
    f.write(b"XRNN")
    f.write(struct.pack("<I", 1))
    ft_bias = model.ft.weight.data.mean(dim=0).detach().cpu().numpy()
    for val in ft_bias: f.write(struct.pack("<h", int(np.clip(round(val * 127.0), -32768, 32767))))
    ft_w = model.ft.weight.data.detach().cpu().numpy()
    for i in range(65536):
        for val in ft_w[i]: f.write(struct.pack("<h", int(np.clip(round(val * 127.0), -32768, 32767))))
    hw = model.hidden.weight.data.detach().cpu().numpy()
    for i in range(32):
        for val in hw[i]: f.write(struct.pack("b", int(np.clip(round(val * 64.0), -128, 127))))
    hb = model.hidden.bias.data.detach().cpu().numpy()
    for val in hb: f.write(struct.pack("<i", int(np.clip(round(val * 127.0 * 64.0), -2147483648, 2147483647))))
    ow = model.output.weight.data[0].detach().cpu().numpy()
    for val in ow: f.write(struct.pack("b", int(np.clip(round(val * 64.0), -128, 127))))
    ob = float(model.output.bias.data[0].detach().cpu().numpy())
    f.write(struct.pack("<i", int(np.clip(round(ob * 64.0 * 64.0 * 400.0), -2147483648, 2147483647))))
    f.write(struct.pack("<i", 16))

print(f"✅ Exported {out_path} ({os.path.getsize(out_path)/(1024*1024):.2f} MB)")
print(f"📊 Best Test MSE: {best_test_loss:.6f} (MAE: {(best_test_loss**0.5)*400:.1f} cp)")

if HF_TOKEN:
    api = HfApi()
    api.upload_file(
        path_or_fileobj=out_path,
        path_in_repo="weights/nnue_weights_community.bin",
        repo_id=repo_id,
        repo_type="dataset",
        token=HF_TOKEN
    )
    print("✅ Uploaded Community NNUE Model to HuggingFace Hub!")
else:
    print("⚠️ HF_TOKEN chưa cấu hình — trọng số chỉ lưu cục bộ")
